In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

DATA_DIR = Path.home() / "kkbox-churn" / "data" / "parquet"

master = pd.read_parquet(DATA_DIR / "master_dataset.parquet")

# Drop non-features
drop_cols = [
    'msno', 'last_transaction_date', 'last_expire_date',
    'last_listen_date', 'first_listen_date'
]
master = master.drop(columns=drop_cols)

# Encode categoricals
for col in ['city', 'registered_via', 'bd_bucket', 'auto_renew_switch']:
    master[col] = master[col].astype('category')

print(f"Shape: {master.shape}")
print(f"Churn rate: {master['is_churn'].mean():.4f}")

Shape: (970960, 102)
Churn rate: 0.0899


In [7]:
# Cell 2 — Temporal split using last_expire_date
# Reload master with dates to get last_expire_date for splitting
master_raw = pd.read_parquet(DATA_DIR / "master_dataset.parquet")

# Check last_expire_date distribution
print(master_raw['last_expire_date'].dtype)
print(master_raw['last_expire_date'].describe())
print(master_raw['last_expire_date'].value_counts().sort_index().tail(20))

datetime64[us]
count                        970960
mean     2017-04-20 08:52:55.723820
min             1970-01-01 00:00:00
25%             2017-04-08 00:00:00
50%             2017-04-18 00:00:00
75%             2017-04-27 00:00:00
max             2023-01-12 00:00:00
Name: last_expire_date, dtype: object
last_expire_date
2019-07-05    1
2019-07-24    1
2019-11-27    1
2019-12-28    1
2020-02-03    1
2020-03-08    1
2020-03-09    1
2020-03-11    1
2020-03-24    1
2020-04-08    1
2020-05-03    1
2020-06-25    1
2020-07-23    1
2020-08-05    1
2020-08-26    1
2020-09-25    1
2020-10-17    1
2021-02-13    1
2021-11-24    1
2023-01-12    1
Name: count, dtype: int64


In [8]:
# Cell 2 — Temporal split on last_expire_date
expire_dates = master_raw['last_expire_date'].copy()

# Find 80th percentile cutoff
cutoff = expire_dates.quantile(0.80)
print(f"Split cutoff (80th percentile): {cutoff}")

train_idx = expire_dates[expire_dates <= cutoff].index
val_idx   = expire_dates[expire_dates > cutoff].index

print(f"Train size: {len(train_idx):,} ({len(train_idx)/len(expire_dates):.1%})")
print(f"Val size:   {len(val_idx):,} ({len(val_idx)/len(expire_dates):.1%})")

X = master.drop(columns=['is_churn'])
y = master['is_churn']

X_train, y_train = X.loc[train_idx], y.loc[train_idx]
X_val,   y_val   = X.loc[val_idx],   y.loc[val_idx]

print(f"\nTrain churn rate: {y_train.mean():.4f}")
print(f"Val churn rate:   {y_val.mean():.4f}")

Split cutoff (80th percentile): 2017-04-30 00:00:00
Train size: 868,856 (89.5%)
Val size:   102,104 (10.5%)

Train churn rate: 0.0709
Val churn rate:   0.2520


In [9]:
# Cell 4 — Fixed params
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,
    'learning_rate': 0.01,
    'num_leaves': 63,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'scale_pos_weight': 10,  # handles imbalance explicitly
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

model = lgb.LGBMClassifier(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)

val_preds = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_preds)
print(f"\nTemporal Val AUC: {auc:.4f}")
print(f"Best iteration: {model.best_iteration_}")

Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.908069
Early stopping, best iteration is:
[7]	valid_0's auc: 0.930797

Temporal Val AUC: 0.9308
Best iteration: 7


In [10]:
# Cell 5 — Diagnose the split
print("Val expire date distribution:")
print(master_raw.loc[val_idx, 'last_expire_date'].value_counts().sort_index().head(20))

print(f"\nVal expire date range: {master_raw.loc[val_idx, 'last_expire_date'].min()} to {master_raw.loc[val_idx, 'last_expire_date'].max()}")
print(f"Train expire date range: {master_raw.loc[train_idx, 'last_expire_date'].min()} to {master_raw.loc[train_idx, 'last_expire_date'].max()}")

# Check churn rate by expire date bucket in full dataset
master_raw['expire_month'] = master_raw['last_expire_date'].dt.to_period('M')
print(f"\nChurn rate by expire month:")
print(master_raw.groupby('expire_month')['is_churn'].agg(['mean','count']).sort_index().to_string())

Val expire date distribution:
last_expire_date
2017-05-01    1459
2017-05-02    3758
2017-05-03    3788
2017-05-04    3637
2017-05-05    2894
2017-05-06    2971
2017-05-07    2961
2017-05-08    2676
2017-05-09    2801
2017-05-10    2649
2017-05-11    2637
2017-05-12    2639
2017-05-13    2402
2017-05-14    2701
2017-05-15    2667
2017-05-16    2586
2017-05-17    2609
2017-05-18    2504
2017-05-19    2553
2017-05-20    2513
Name: count, dtype: int64

Val expire date range: 2017-05-01 00:00:00 to 2023-01-12 00:00:00
Train expire date range: 1970-01-01 00:00:00 to 2017-04-30 00:00:00

Churn rate by expire month:
                  mean   count
expire_month                  
1970-01       1.000000       7
2013-12       1.000000       1
2014-04       1.000000       1
2015-11       1.000000       1
2015-12       1.000000     110
2016-01       0.993151     146
2016-02       1.000000     554
2016-03       0.996840     633
2016-04       1.000000      83
2016-05       1.000000      85
2016-06    

In [11]:
# Cell 6 — Correct temporal split: April train, May val
apr_idx = master_raw[master_raw['last_expire_date'].dt.to_period('M') == '2017-04'].index
may_idx = master_raw[master_raw['last_expire_date'].dt.to_period('M') == '2017-05'].index

X = master.drop(columns=['is_churn'])
y = master['is_churn']

X_train, y_train = X.loc[apr_idx], y.loc[apr_idx]
X_val,   y_val   = X.loc[may_idx], y.loc[may_idx]

print(f"Train (April): {len(X_train):,}, churn rate: {y_train.mean():.4f}")
print(f"Val   (May):   {len(X_val):,},   churn rate: {y_val.mean():.4f}")

Train (April): 800,236, churn rate: 0.0165
Val   (May):   79,942,   churn rate: 0.0502


In [12]:
# Cell 7 — LightGBM on correct temporal split
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'is_unbalance': True,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

model = lgb.LGBMClassifier(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)

val_preds = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_preds)
print(f"\nTemporal Val AUC: {auc:.4f}")
print(f"Best iteration: {model.best_iteration_}")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.939849
[200]	valid_0's auc: 0.946018
Early stopping, best iteration is:
[221]	valid_0's auc: 0.946788

Temporal Val AUC: 0.9468
Best iteration: 221


In [13]:
# Cell 8 — Feature importance
feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(feat_imp.head(20).to_string())

                                feature  importance
4           days_since_last_transaction        1972
7                        days_to_expiry        1754
35                                 city         514
3                current_payment_method         365
33                            t1_amount         296
41                     account_age_days         295
92  days_between_last_listen_and_expiry         285
31                    avg_interval_days         258
5                days_since_last_cancel         253
63                    active_days_trend         252
37                             bd_clean         242
62                listening_trend_ratio         227
6                  customer_tenure_days         206
69                       final_day_secs         204
59                  completion_rate_15d         202
99                auto_not_cancel_ratio         197
60                      unq_per_day_all         193
87                        skip_rate_all         180
58          

In [15]:
# Cell 9 — XGBoost only
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import time

X_train_xgb = X_train.copy()
X_val_xgb   = X_val.copy()
for col in ['city','registered_via','bd_bucket','auto_renew_switch']:
    X_train_xgb[col] = X_train_xgb[col].cat.codes
    X_val_xgb[col]   = X_val_xgb[col].cat.codes

t0 = time.time()
xgb = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=10,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    early_stopping_rounds=50,
    eval_metric='auc'
)
xgb.fit(X_train_xgb, y_train, eval_set=[(X_val_xgb, y_val)], verbose=False)
xgb_preds = xgb.predict_proba(X_val_xgb)[:, 1]
xgb_auc = roc_auc_score(y_val, xgb_preds)
print(f"XGBoost AUC: {xgb_auc:.4f} | best iter: {xgb.best_iteration} | time: {time.time()-t0:.0f}s")

XGBoost AUC: 0.8383 | best iter: 351 | time: 11s


In [ ]:
# Cell 10 — CatBoost
from catboost import CatBoostClassifier
import time

X_train_cb = X_train.copy()
X_val_cb   = X_val.copy()

cat_features = ['city', 'registered_via', 'bd_bucket', 'auto_renew_switch']
for col in cat_features:
    X_train_cb[col] = X_train_cb[col].astype(object).fillna('missing').astype(str)
    X_val_cb[col]   = X_val_cb[col].astype(object).fillna('missing').astype(str)

t0 = time.time()
cb = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    random_seed=42,
    early_stopping_rounds=50,
    class_weights=[1, 10],
    verbose=False,
    cat_features=cat_features
)
cb.fit(X_train_cb, y_train, eval_set=(X_val_cb, y_val))
cb_preds = cb.predict_proba(X_val_cb)[:, 1]
cb_auc = roc_auc_score(y_val, cb_preds)
print(f"CatBoost AUC: {cb_auc:.4f} | best iter: {cb.best_iteration_} | time: {time.time()-t0:.0f}s")

CatBoost AUC: 0.8954 | best iter: 27 | time: 10s
